## Simple RAG Demo with Local Models

This notebook demonstrates a minimal **Retrieval-Augmented Generation (RAG)** pipeline on a small "Company Policy Manual" text file.

We will:
- **Chunk** a plain-text knowledge base (`my_knowledge.txt`)
- **Embed** the chunks with `sentence-transformers`
- **Index** them with `FAISS` for similarity search
- **Generate answers** with an instruction-tuned model (`google/flan-t5-base`)

You can run everything end-to-end on a laptop and commit this notebook as a self-contained demo repo.

In [6]:
!pip install transformers sentence-transformers faiss-cpu langchain

## 1. Install and import dependencies

This cell installs the core libraries we need:
- `transformers` and `sentence-transformers` for embeddings and generation
- `faiss-cpu` for vector search
- `langchain` for convenient text splitting

You typically run this once per environment (or put it into a `requirements.txt` when turning this into a project).

## 2. Load and chunk the knowledge base

We load the contents of `my_knowledge.txt` and split it into overlapping chunks using LangChain's `RecursiveCharacterTextSplitter`.

This makes it easier for the model to retrieve only the most relevant parts of the document for a given question.

In [7]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load the text file:
with open("my_knowledge.txt", "r") as f:
    knowledge_text = f.read()

# 1. Initialize the Text Splitter
# This splitter is smart. It tries to split on paragraphs ("\n\n"),
# then newlines ("\n"), then spaces (" "), to keep semantically related text together as much as possible.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 150, # Max size of the chunk
    chunk_overlap = 20, # Overlap to maintain context between chunks
    length_function = len
)

# 2. Create the chunks:
chunks = text_splitter.split_text(knowledge_text)

print(f"We have {len(chunks)} chunks here:")
for i, chunk in enumerate(chunks):
    print(f"---- Chunk {i+1} ----\n{chunk}\n")

We have 5 chunks here:
---- Chunk 1 ----
Company Policy Manual:

---- Chunk 2 ----
- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays

---- Chunk 3 ----
Thursdays. Mondays and Fridays are optional remote days.

---- Chunk 4 ----
- PTO Policy: Full-time employees receive 20 days of Paid Time Off (PTO) per year. PTO accrues monthly.

---- Chunk 5 ----
- Tech Stack: The official backend language is Python, and the official frontend framework is React. For mobile development, we use React Native.



## 3. Create dense embeddings for each chunk

We encode every text chunk into a dense vector using the `all-MiniLM-L6-v2` model from `sentence-transformers`.

These embeddings will later be used to find the most similar chunks to a user question.

In [8]:
from sentence_transformers import SentenceTransformer

# 1. Load the embedding model
# 'all-MiniLM-L6-v2' is a fantastic, fast, and small model.
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Embed all our chunks:
chunk_embeddings = model.encode(chunks)

print(f"Shape of the embeddings: {chunk_embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Shape of the embeddings: (5, 384)


## 4. Build a FAISS index for similarity search

We create a simple L2-based FAISS index over the chunk embeddings.

At query time we will search this index to retrieve the top-
k most relevant chunks to feed into the language model.

In [9]:
import faiss
import numpy as np

# Get the dimensions of our vectors:
d = chunk_embeddings.shape[1]

# 1. Create a FAISS index
# IndexFlatL2 is the simplest, most basic index. It calculates the exact distance (L2 distance) between our query and all vectors

index = faiss.IndexFlatL2(d)

# 2. Add our chunk embeddings to the index, We must convert to float32 for FAISS
index.add(np.array(chunk_embeddings).astype('float32'))

print(f"FAISS index has been created with {index.ntotal} vectors")

FAISS index has been created with 5 vectors


## 5. Define the RAG answer function

We now load an instruction-tuned encoder–decoder model (`google/flan-t5-base`) and wrap it in a small helper:
- First we **retrieve** the most similar chunk(s) from the FAISS index
- Then we **augment** the user question with that context
- Finally we **generate** a short, grounded answer using the model

The `answer_question` function is the main entry point you can call from other code.

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load a stronger open instruction-tuned seq2seq model
# This model is fully open and doesn't require authentication.
llm_model_name = "google/flan-t5-base"
llm_tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

def generate_answer(prompt: str, max_new_tokens: int = 128) -> str:
    inputs = llm_tokenizer(prompt, return_tensors="pt")
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,   # deterministic
        temperature=0.0,
    )
    return llm_tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- This is our RAG pipeline function ---
def answer_question(query: str) -> str:
    """Answer a question using only retrieved context from FAISS."""
    # 1. RETRIEVE: embed the user's query and search FAISS
    query_embedding = model.encode([query]).astype("float32")

    # Keep context tight so the model focuses on the most relevant chunk
    k = 1
    distances, indices = index.search(query_embedding, k)

    # Get the actual text chunks from our original 'chunks' list
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # 2. AUGMENT: build a clear, constrained prompt
    prompt = f"""You are a helpful assistant.
Use ONLY the information inside <context> tags.
If the answer is not in the context, reply exactly: "I don't have that information."

<context>
{context}
</context>

Question: {query}
Answer in one short, direct sentence:
"""

    # 3. GENERATE: call the seq2seq model
    answer_text = generate_answer(prompt, max_new_tokens=64)
    print(f"---- CONTEXT ----\n{context}\n")
    return answer_text.strip()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## 6. Ask questions against the knowledge base

Finally, we query the RAG pipeline with a few example questions.

- When the answer **is** present in the context, the model should restate it clearly.
- When the answer **is not** present, it should reply exactly: `"I don't have that information."`

You can add more questions here or adapt this notebook into a script or API.

In [11]:
query1 = "What is the WFH policy?"
print(f"Query: {query1}")
print(f"Answer: {answer_question(query1)}\n")

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Query: What is the WFH policy?
---- CONTEXT ----
- WFH Policy: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays. Mondays

Answer: All employees are eligible for a hybrid WFH schedule. Employees must be in the office on Tuesdays, Wednesdays, and Thursdays.



In [12]:
query_2 = "What is the company's dental plan?"
print(f"Query: {query_2}")
print(f"Answer: {answer_question(query_2)}\n")

Query: What is the company's dental plan?
---- CONTEXT ----
Company Policy Manual:

Answer: I don't have that information.

